In [ ]:
import os
import sys
from pathlib import Path

# Ensure we're in the project root
project_root = Path("/Users/arnavhmutt/Desktop/aviation-ds-projects/project-2-predictive-maintenance").resolve()
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print(f"Working directory: {os.getcwd()}")
print(f"Project root in sys.path: {str(project_root) in sys.path}")

# Import modules
from data.load import load_dataset, load_config
from data.preprocess import preprocess_train, preprocess_test
from features.health_index import build_health_index
from features.velocity import build_velocity
from features.variability import build_variability
from model.clustering import build_clustering
from model.risk import build_risk_score

print("✅ All imports successful\n")

# Load and run pipeline
config = load_config(project_root / "config" / "config.yaml")
config["dataset"]["raw_path"] = str(project_root / "data" / "raw")

train_raw, test_raw, _ = load_dataset(config)
train_proc, scaler, sensor_cols = preprocess_train(train_raw, config)
test_proc = preprocess_test(test_raw, config, scaler)
train_hi, test_hi, hi_art = build_health_index(train_proc, test_proc, config)
train_vel, test_vel, velocity_artifacts = build_velocity(train_hi, test_hi, config)
train_var, test_var, var_art = build_variability(train_vel, test_vel, config)
train_cl, test_cl, cl_art = build_clustering(train_var, test_var, config)
train_rs, test_rs, risk_art = build_risk_score(train_cl, test_cl, cl_art)

# Shape checks
assert train_rs.shape[0] == 20631
assert test_rs.shape[0] == 13096
assert "risk_score" in train_rs.columns

# Range checks
assert train_rs["risk_score"].between(0.0, 1.0).all()
assert test_rs["risk_score"].between(0.0, 1.0).all()

# Risk must increase with degradation
mean_risk = train_rs.groupby("risk_state")["risk_score"].mean()
assert mean_risk["Critical"] > mean_risk["Degrading"] > mean_risk["Healthy"], \
    f"Risk ordering violated: {mean_risk.to_dict()}"

# No NaN
assert train_rs["risk_score"].isna().sum() == 0
assert test_rs["risk_score"].isna().sum() == 0

print("✅ All checks passed")
print("\nRisk score by cluster (train):")
print(mean_risk.round(4))
print("\nOverall risk score stats (train):")
print(train_rs["risk_score"].describe().round(4))
print("\nRisk artifacts:")
print(f"   d_min: {risk_art.d_min:.4f}")
print(f"   d_max: {risk_art.d_max:.4f}")

Changed to: /Users/arnavhmutt/Desktop/aviation-ds-projects/project-2-predictive-maintenance
sys.path[0] before: /Users/arnavhmutt/Desktop/aviation-ds-projects/project-2-predictive-maintenance
sys.path[0] after: /Users/arnavhmutt/Desktop/aviation-ds-projects/project-2-predictive-maintenance
Can find data/load.py: True


ModuleNotFoundError: No module named 'data.load'